In [1]:
# Imports
import pandas as pd
import numpy as np


In [2]:
# Load train and validation sets, parsing Time as datetime
train_df = pd.read_csv('../Data/train.csv')
val_df = pd.read_csv('../Data/val.csv')

train_df['Time'] = pd.to_datetime(train_df['Time'], format='mixed', errors='coerce')
val_df['Time'] = pd.to_datetime(val_df['Time'], format='mixed', errors='coerce')


In [3]:
# Sort by Time to guarantee chronological order before creating lag features
train_df = train_df.sort_values('Time').reset_index(drop=True)
val_df = val_df.sort_values('Time').reset_index(drop=True)


In [4]:
# Create lag and rolling features from Power[kW] on each DataFrame separately
dfs = [train_df, val_df]
for df in dfs:
    df['power_lag1'] = df['Power[kW]'].shift(1)
    df['power_lag2'] = df['Power[kW]'].shift(2)
    df['power_lag4'] = df['Power[kW]'].shift(4)
    df['power_roll4'] = df['Power[kW]'].rolling(window=4).mean()


In [5]:
# Create forecasting targets, then invalidate ones where the shifted row is not genuinely 15/60 minutes ahead
# (daylight-only data has large gaps at day boundaries, so shift(-1)/shift(-4) can land on the next day)
for df in dfs:
    time_diff_1 = df['Time'].shift(-1) - df['Time']
    time_diff_4 = df['Time'].shift(-4) - df['Time']

    df['target_15min_buggy'] = df['Power[kW]'].shift(-1)
    df['target_1hour_buggy'] = df['Power[kW]'].shift(-4)

    df['target_15min'] = df['target_15min_buggy'].copy()
    df['target_1hour'] = df['target_1hour_buggy'].copy()

    df.loc[time_diff_1 != pd.Timedelta(minutes=15), 'target_15min'] = np.nan
    df.loc[time_diff_4 != pd.Timedelta(minutes=60), 'target_1hour'] = np.nan

train_df_pre_fix = train_df.copy()
val_df_pre_fix = val_df.copy()


In [6]:
# Drop rows with NaN targets (day-boundary mismatches now removed here too, not just tail rows)
train_rows_before = len(train_df)
val_rows_before = len(val_df)

train_boundary_dropped = ((train_df['target_15min'].isna() & train_df['target_15min_buggy'].notna())
                           | (train_df['target_1hour'].isna() & train_df['target_1hour_buggy'].notna())).sum()
val_boundary_dropped = ((val_df['target_15min'].isna() & val_df['target_15min_buggy'].notna())
                         | (val_df['target_1hour'].isna() & val_df['target_1hour_buggy'].notna())).sum()

subset_cols = ['power_lag1', 'power_lag2', 'power_lag4', 'power_roll4', 'target_15min', 'target_1hour']
train_df = train_df.dropna(subset=subset_cols).reset_index(drop=True)
val_df = val_df.dropna(subset=subset_cols).reset_index(drop=True)

train_rows_after = len(train_df)
val_rows_after = len(val_df)

print('Train rows before:', train_rows_before, 'after:', train_rows_after)
print('Val rows before:', val_rows_before, 'after:', val_rows_after)
print('Train rows newly dropped due to day-boundary mismatches (not tail):', train_boundary_dropped)
print('Val rows newly dropped due to day-boundary mismatches (not tail):', val_boundary_dropped)


Train rows before: 71647 after: 65823
Val rows before: 17735 after: 16299
Train rows newly dropped due to day-boundary mismatches (not tail): 5816
Val rows newly dropped due to day-boundary mismatches (not tail): 1428


In [7]:
# Verify alignment: target_15min at row N equals Power[kW] at row N+1, target_1hour at row N equals Power[kW] at row N+4
check_cols = ['Time', 'Power[kW]', 'power_lag1', 'target_15min', 'target_1hour']
print(train_df[check_cols].head(6))


                 Time  Power[kW]  power_lag1  target_15min  target_1hour
0 2017-01-01 08:15:00      0.132       0.020         0.176         0.332
1 2017-01-01 08:30:00      0.176       0.132         0.244         0.276
2 2017-01-01 08:45:00      0.244       0.176         0.260         0.392
3 2017-01-01 09:00:00      0.260       0.244         0.332         0.552
4 2017-01-01 09:15:00      0.332       0.260         0.276         0.644
5 2017-01-01 09:30:00      0.276       0.332         0.392         0.476


In [8]:
# Regression metrics helper
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def regression_report(y_true, y_pred, label=''):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    print(f'{label}  RMSE={rmse:,.3f}  MAE={mae:,.3f}  R2={r2:.3f}')
    return rmse, mae, r2


In [9]:
# Save the prepared forecasting datasets with the corrected targets (drop the temporary buggy columns first)
train_df = train_df.drop(columns=['target_15min_buggy', 'target_1hour_buggy'])
val_df = val_df.drop(columns=['target_15min_buggy', 'target_1hour_buggy'])

train_df.to_csv('../Data/train_forecast.csv', index=False)
val_df.to_csv('../Data/val_forecast.csv', index=False)


In [10]:
# Print final shape and column list for both saved DataFrames
print('train_forecast shape:', train_df.shape)
print('train_forecast columns:', list(train_df.columns))
print('val_forecast shape:', val_df.shape)
print('val_forecast columns:', list(val_df.columns))


train_forecast shape: (65823, 25)
train_forecast columns: ['Time', 'GHI', 'temp', 'pressure', 'humidity', 'wind_speed', 'rain_1h', 'snow_1h', 'clouds_all', 'weather_type', 'sunlightTime', 'dayLength', 'SunlightTime/daylength', 'hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'isSun', 'Power[kW]', 'power_lag1', 'power_lag2', 'power_lag4', 'power_roll4', 'target_15min', 'target_1hour']
val_forecast shape: (16299, 25)
val_forecast columns: ['Time', 'GHI', 'temp', 'pressure', 'humidity', 'wind_speed', 'rain_1h', 'snow_1h', 'clouds_all', 'weather_type', 'sunlightTime', 'dayLength', 'SunlightTime/daylength', 'hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'isSun', 'Power[kW]', 'power_lag1', 'power_lag2', 'power_lag4', 'power_roll4', 'target_15min', 'target_1hour']


In [11]:
# Persistence baseline with the corrected targets, compared against the previous (buggy) targets
val_buggy_15min = val_df_pre_fix.dropna(subset=['target_15min_buggy'])
val_buggy_1hour = val_df_pre_fix.dropna(subset=['target_1hour_buggy'])

rmse_15min_buggy, mae_15min_buggy, r2_15min_buggy = regression_report(val_buggy_15min['target_15min_buggy'], val_buggy_15min['Power[kW]'], label='15-min persistence baseline (buggy, previous)')
rmse_1hour_buggy, mae_1hour_buggy, r2_1hour_buggy = regression_report(val_buggy_1hour['target_1hour_buggy'], val_buggy_1hour['Power[kW]'], label='1-hour persistence baseline (buggy, previous)')

rmse_15min, mae_15min, r2_15min = regression_report(val_df['target_15min'], val_df['Power[kW]'], label='15-min persistence baseline (fixed)')
rmse_1hour, mae_1hour, r2_1hour = regression_report(val_df['target_1hour'], val_df['Power[kW]'], label='1-hour persistence baseline (fixed)')


15-min persistence baseline (buggy, previous)  RMSE=1.723  MAE=0.925  R2=0.863
1-hour persistence baseline (buggy, previous)  RMSE=2.894  MAE=1.890  R2=0.614
15-min persistence baseline (fixed)  RMSE=1.796  MAE=0.997  R2=0.854
1-hour persistence baseline (fixed)  RMSE=3.016  MAE=2.032  R2=0.589
